# Visão Computacional — Classificação Wildfire / No-Wildfire

**GS 2026.1 — Space Connect**  
**Aluno:** Moacyr Cabral da Silva — RM 559263  
**Disciplina:** Visão Computacional  
**Atividade:** GS - Visão Computacional  
**Entrega:** 2026-06-09

Este notebook implementa um classificador binário de imagens de satélite (Wildfire vs. No-Wildfire) usando transfer learning com MobileNetV2. O modelo apoia o monitoramento de risco de queimadas e degradação vegetal na plataforma Space Connect.

**Etapas:** entendimento do problema → EDA → pré-processamento → modelagem → avaliação → proposta de integração.

## 1. Contexto do problema ambiental

O monitoramento de áreas de preservação e supressão vegetal em escala continental é inviável apenas com fiscalização presencial. O sensoriamento remoto via satélite, combinado com modelos de Visão Computacional, permite triagem rápida de regiões de risco para priorizar fiscalização e ações preventivas.

Este notebook treina um modelo binário sobre imagens 350x350 obtidas a partir de coordenadas com e sem ocorrência de incêndio florestal. O foco da avaliação está em **falsos negativos**: áreas de risco que o modelo deixaria passar — o erro mais caro neste contexto.

## 2. Setup do ambiente e Kaggle API

Para baixar o Wildfire Prediction Dataset:

1. Acesse https://www.kaggle.com/settings/account → "Create New Token" → baixa um arquivo `kaggle.json`.
2. Execute a célula abaixo — ela pede o upload do `kaggle.json` na sessão atual do Colab.

In [ ]:
from google.colab import files
import os, json, pathlib

uploaded = files.upload()  # selecione o kaggle.json

kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / 'kaggle.json').write_bytes(uploaded['kaggle.json'])
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('Kaggle API configurada.')

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d abdelghaniaaba/wildfire-prediction-dataset
!unzip -q wildfire-prediction-dataset.zip -d /content/wildfire
!ls /content/wildfire

## 3. Análise exploratória (EDA)

Verifica a estrutura de pastas (treino/validação/teste), contagens por classe e amostras visuais.

In [ ]:
import os, pathlib
import matplotlib.pyplot as plt
from PIL import Image

BASE = pathlib.Path('/content/wildfire')
for split in ['train', 'valid', 'test']:
    for cls in ['wildfire', 'nowildfire']:
        d = BASE / split / cls
        n = len(list(d.glob('*.jpg'))) if d.exists() else 0
        print(f'{split:>5} / {cls:>10}: {n} imagens')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, cls in enumerate(['wildfire', 'nowildfire']):
    paths = list((BASE / 'train' / cls).glob('*.jpg'))[:4]
    for ax, p in zip(axes[col], paths):
        ax.imshow(Image.open(p)); ax.set_title(cls); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Subset de 20% para a primeira rodada

O dataset completo tem ~42.850 imagens. Para uma primeira rodada rápida em T4 gratuita, treinamos com 20% das imagens em cada split. Se sobrar tempo, retreinamos com 100%.

In [ ]:
import random, shutil
random.seed(42)

SUBSET = pathlib.Path('/content/wildfire_subset')
FRAC = 0.2
if SUBSET.exists():
    shutil.rmtree(SUBSET)

for split in ['train', 'valid', 'test']:
    for cls in ['wildfire', 'nowildfire']:
        src = BASE / split / cls
        dst = SUBSET / split / cls
        dst.mkdir(parents=True, exist_ok=True)
        files = list(src.glob('*.jpg'))
        sample = random.sample(files, k=int(len(files) * FRAC))
        for f in sample:
            os.symlink(f, dst / f.name)
        print(f'{split}/{cls}: {len(sample)} imagens (de {len(files)})')

## 5. Pipeline de dados com `tf.data`

Resize para 224x224 (entrada nativa do MobileNetV2), normalização via `preprocess_input` do MobileNetV2, e data augmentation apenas no treino.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

IMG = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        SUBSET / split,
        labels='inferred',
        label_mode='binary',
        class_names=['nowildfire', 'wildfire'],
        image_size=(IMG, IMG),
        batch_size=BATCH,
        shuffle=shuffle,
        seed=42,
    )

train_ds = make_ds('train', True)
val_ds   = make_ds('valid', False)
test_ds  = make_ds('test',  False)

augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

def prep(ds, training):
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(AUTOTUNE)

train_ds = prep(train_ds, True)
val_ds   = prep(val_ds, False)
test_ds  = prep(test_ds, False)

## 6. Modelo: MobileNetV2 com transfer learning

Base congelada (pesos ImageNet) + `GlobalAveragePooling` + `Dropout` + `Dense(1, sigmoid)`. Justificativa: arquitetura leve, treina rápido em T4 e reconhecida pelo enunciado.

In [ ]:
base = MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = tf.keras.Input(shape=(IMG, IMG, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)
model.summary()

## 7. Treinamento

Fase 1 (head): 5 épocas com base congelada. Fase 2 (fine-tuning leve): 3 épocas descongelando os últimos blocos.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True, monitor='val_auc', mode='max'),
]

hist1 = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)

In [ ]:
# Fine-tuning leve
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=3, callbacks=callbacks)

## 8. Avaliação

Métricas obrigatórias para classificação binária: acurácia, precisão, recall, F1, matriz de confusão e ROC AUC.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

y_true, y_prob = [], []
for x, y in test_ds:
    y_true.append(y.numpy().ravel())
    y_prob.append(model.predict(x, verbose=0).ravel())
y_true = np.concatenate(y_true)
y_prob = np.concatenate(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=['nowildfire', 'wildfire'], digits=4))
print('ROC AUC:', round(roc_auc_score(y_true, y_prob), 4))
cm = confusion_matrix(y_true, y_pred)
print('Matriz de confusão:')
print(cm)
print(f'Falsos negativos (wildfire perdidos): {cm[1,0]}')
print(f'Falsos positivos (alarme falso):     {cm[0,1]}')

In [ ]:
import matplotlib.pyplot as plt
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_true, y_prob):.3f}')
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC — Wildfire/No-Wildfire'); plt.legend(); plt.show()

## 9. Análise dos erros

**Falsos negativos** representam áreas de risco que o modelo classificaria como seguras — o erro mais caro no contexto ambiental. Em uma operação real, ajustar o threshold para baixo (ex. 0.35 em vez de 0.5) aumenta o recall na classe wildfire ao custo de mais falsos positivos, deslocando a triagem para um regime mais conservador.

**Falsos positivos** geram custo de fiscalização (visita a campo desnecessária), mas são preferíveis a um falso negativo no contexto de prevenção.

In [ ]:
# Curva precisão x recall por threshold para apoiar a discussão
from sklearn.metrics import precision_recall_curve
p, r, t = precision_recall_curve(y_true, y_prob)
plt.figure(figsize=(6,5))
plt.plot(t, p[:-1], label='precisão')
plt.plot(t, r[:-1], label='recall')
plt.xlabel('threshold'); plt.title('Precisão e recall por threshold (classe wildfire)')
plt.legend(); plt.show()

## 10. Proposta de integração com a plataforma Space Connect

O modelo treinado é exportado e exposto como endpoint de inferência consumido pelo front-end React+Vite (disciplina 09). Fluxo:

1. Recorte de imagem de satélite (350x350) chega via API ou job agendado da automação RPA (disciplina 07).
2. Imagem passa pelo classificador → probabilidade + classe.
3. Resultado vai para o dashboard com camada de mapa (Front-End) e dispara alerta se probabilidade > threshold operacional.
4. Estação IoT (disciplina 05) cruza alerta de imagem com leituras locais (temperatura, radiação, perda de comunicação) para escalar prioridade.
5. Chatbots (NLP/GenAI) respondem dúvidas operacionais sobre a área marcada usando o corpus técnico.

Limitações: o modelo é binário e não localiza o foco dentro da imagem; resolução 350x350 ignora detalhes finos; e o dataset Kaggle, embora amplo, não cobre todas as biomas brasileiras.

In [ ]:
# Exporta o modelo para uso futuro pelo front-end / API
model.save('/content/wildfire_mobilenetv2.keras')
print('Modelo salvo em /content/wildfire_mobilenetv2.keras')

## 11. Próximos passos

- Reexecutar com 100% do dataset se houver tempo de Colab restante.
- Calibrar threshold em função do custo operacional de falsos negativos.
- Avaliar Grad-CAM para mostrar regiões da imagem que mais contribuem para a decisão (entra no relatório como evidência de explicabilidade).